# 03 — Modèle ML : Gradient Boosted Trees + SHAP sur les faits de jeu

Objectif : entraîner un modèle qui prédit la victoire à partir des faits de jeu (objectifs, combat, early game...) réalisés par chaque champion, puis l'expliquer avec SHAP pour répondre à trois questions : quels faits comptent pour gagner en général, lesquels comptent pour un champion donné, et lesquels priment pour une composition donnée.

**Mise à jour** : `data/matches.db` contient maintenant **12 482 matchs** (collecte à grande échelle terminée le 2026-09-15/16) — le pipeline ci-dessous a été validé d'abord sur un petit échantillon (~20 matchs), puis rejoué tel quel sur les vraies données.

**Partie 1 : extraire un dataset tabulaire propre depuis SQLite.**

## Concept 1 — Charger les matchs bruts

On réutilise `lol_assistant.database.get_connection` (même base que la phase 2), puis `pd.read_sql_query` pour récupérer toutes les lignes de la table `matches` sous forme de `DataFrame` — chaque ligne a `match_id`, `region`, `raw_json` (le JSON complet du match, tel que stocké par le crawler).

In [1]:
import pandas as pd

from lol_assistant.database import get_connection

conn = get_connection("../data/matches.db")

raw_df = pd.read_sql_query("SELECT match_id, raw_json FROM matches", conn)
print("Nombre de matchs bruts:", len(raw_df))
raw_df.head()

Nombre de matchs bruts: 12482


,match_id,raw_json
0,EUW1_7981560267,"{""metadata"": {""dataVersion"": ""2"", ""matchId"": ""..."
1,EUW1_7981526567,"{""metadata"": {""dataVersion"": ""2"", ""matchId"": ""..."
2,EUW1_7979271688,"{""metadata"": {""dataVersion"": ""2"", ""matchId"": ""..."
3,EUW1_7978928820,"{""metadata"": {""dataVersion"": ""2"", ""matchId"": ""..."
4,EUW1_7978885105,"{""metadata"": {""dataVersion"": ""2"", ""matchId"": ""..."


## Concept 2 — Extraire une ligne structurée par match

`raw_json` est un gros bloc de texte JSON — on le parse avec `json.loads`, puis on isole ce dont on a besoin : la liste des 5 champions de l'équipe bleue, celle des 5 champions de l'équipe rouge, si l'équipe bleue a gagné (`teamId == 100`), et `gameDuration` (utile juste après pour repérer les remakes). On garde aussi `gameMode` — utile pour un autre nettoyage qu'on va faire juste après (exclure les modes de jeu qui ne sont pas le 5v5 classique).

In [2]:
import json


def extract_match_row(raw_json):
    match_data = json.loads(raw_json)
    info = match_data["info"]

    blue_champions = []
    red_champions = []
    blue_win = None

    for p in info["participants"]:
        if p["teamId"] == 100:
            blue_champions.append(p["championName"])
            blue_win = p["win"]
        else:
            red_champions.append(p["championName"])

    return {
        "blue_champions": blue_champions,
        "red_champions": red_champions,
        "blue_win": blue_win,
        "game_duration": info["gameDuration"],
        "game_mode": info["gameMode"],
    }


rows = [extract_match_row(raw) for raw in raw_df["raw_json"]]
matches_df = pd.DataFrame(rows)
matches_df["match_id"] = raw_df["match_id"]

matches_df.head()

,blue_champions,red_champions,blue_win,game_duration,game_mode,match_id
0,"[Jax, Shyvana, Leblanc, Yunara, Leona]","[Camille, Sylas, Zed, Caitlyn, Morgana]",True,1141,CLASSIC,EUW1_7981560267
1,"[Zilean, Skarner, Tristana, Viktor, Sona]","[Pantheon, Belveth, Riven, Syndra, Elise]",True,1773,CLASSIC,EUW1_7981526567
2,"[Rumble, Sylas, Ahri, Ezreal, Nautilus]","[Aurora, Talon, Zed, Pantheon, Seraphine]",False,2488,CLASSIC,EUW1_7979271688
3,"[Pantheon, Qiyana, AurelionSol, Jhin, Elise]","[Tryndamere, Zyra, Ahri, Sivir, Zilean]",False,1494,CLASSIC,EUW1_7978928820
4,"[Jayce, Skarner, Ekko, Zoe, Thresh]","[Pantheon, LeeSin, Lux, Tristana, Nautilus]",True,2377,CLASSIC,EUW1_7978885105


## Concept 3 — Nettoyage : repérer les remakes

Un "remake" (partie annulée tôt, ex: un joueur absent dès le début, remboursée par Riot) n'est pas un vrai résultat de composition — l'inclure fausserait l'entraînement (une victoire/défaite qui ne reflète rien de stratégique). On repère ça via `gameDuration`, anormalement bas pour une vraie partie.

On inspecte d'abord la distribution avant de choisir un seuil, plutôt que d'en deviner un au hasard — et on vérifie au passage l'unité (secondes ou millisecondes, Riot a changé ça selon les versions de patch).

In [3]:
matches_df["game_duration"].describe()

count    12482.000000
mean      1633.152780
std        409.425716
min         64.000000
25%       1416.000000
50%       1643.000000
75%       1886.000000
max       3488.000000
Name: game_duration, dtype: float64

**À vérifier en exécutant** : si les valeurs sont de l'ordre de quelques centaines à ~3000, `gameDuration` est en **secondes** (une partie dure typiquement 15-40 min, soit 900-2400s) — un seuil de remake autour de 300s (5 min) est raisonnable. Si les valeurs sont de l'ordre du million, c'est en **millisecondes**, et il faudra diviser par 1000 avant de filtrer. On ajustera le code juste après selon ce qu'on observe.

In [4]:
REMAKE_THRESHOLD_SECONDS = 300  # confirmé : gameDuration est en secondes sur cet échantillon

n_before = len(matches_df)
matches_df = matches_df[matches_df["game_duration"] >= REMAKE_THRESHOLD_SECONDS].reset_index(drop=True)
n_after = len(matches_df)

print(f"Matchs retirés (probables remakes) : {n_before - n_after}")
print(f"Matchs restants : {n_after}")

Matchs retirés (probables remakes) : 132
Matchs restants : 12350


## Concept 3bis — Nettoyage : exclure les modes de jeu hors-sujet

Le crawler snowball explore les matchs récents des joueurs croisés — sans distinction de mode. Résultat concret sur nos données : certains matchs sont en **Arena** (`gameMode = "CHERRY"`, équipes de 2, pas de lanes, objectifs complètement différents), pas du 5v5 Summoner's Rift classique. Les inclure fausserait tout : ce ne sont pas des variations stratégiques d'un même jeu, c'est un jeu différent.

On ne garde que `gameMode == "CLASSIC"` (couvre normales, ranked solo/duo, ranked flex, Clash — tout le 5v5 classique). Le crawler (`lol_assistant/crawler.py`) a aussi été corrigé pour ne plus stocker ces matchs à l'avenir.

In [5]:
print("Modes de jeu présents:", matches_df["game_mode"].value_counts().to_dict())

n_before = len(matches_df)
matches_df = matches_df[matches_df["game_mode"] == "CLASSIC"].reset_index(drop=True)
n_after = len(matches_df)

print(f"Matchs retirés (hors 5v5 classique) : {n_before - n_after}")
print(f"Matchs restants : {n_after}")

Modes de jeu présents: {'CLASSIC': 12346, 'CHERRY': 4}
Matchs retirés (hors 5v5 classique) : 4
Matchs restants : 12346


---

**Checkpoint** : à ce stade, `matches_df` contient un dataset tabulaire propre (`blue_champions`, `red_champions`, `blue_win`, `match_id`), débarrassé des remakes éventuels.

**Mise à jour d'architecture (voir mémoire projet)** : on n'utilise plus une Factorization Machine sur les compositions seules — un problème mathématique réel (produit scalaire symétrique, incapable de coder une relation directionnelle "A contre B") nous a fait pivoter vers **Gradient Boosted Trees + SHAP** sur des **faits de jeu** (objectifs, kills, early game...), extraits du champ `challenges` déjà présent dans `Match-V5` (pas besoin de la Timeline).

Prochaine étape : extraire les faits de jeu depuis `challenges`, au niveau équipe.

## Concept 4 — Le champ `challenges` : une mine de faits de jeu déjà agrégés

Chaque participant d'un match `Match-V5` a un champ `challenges` avec ~125 statistiques dérivées, déjà calculées par Riot : objectifs (`dragonTakedowns`, `teamBaronKills`...), combat (`killParticipation`, `teamDamagePercentage`...), early game (`laningPhaseGoldExpAdvantage`...), vision... exactement le genre de "faits de jeu" qu'on voulait extraire des événements bruts, mais sans avoir à toucher à la Timeline.

On exclut d'abord ce qui vient à coup sûr d'autres modes de jeu (Arena `SWARM_*`, quelques mécaniques ARAM spécifiques) — le reste est gardé large, comme convenu.

Ensuite, un détail structurel important : certains champs sont **au niveau équipe** (répétés à l'identique chez les 5 joueurs, ex: `teamBaronKills`), d'autres **au niveau joueur** (varient, ex: `soloKills`). On garde les deux catégories **séparément identifiées**, mais **sans jamais sommer les champs individuels** — l'objectif étant justement de savoir quel fait de jeu compte pour quel champion précisément, sommer effacerait cette information. Les champs équipe restent un contexte partagé ; les champs joueur restent attachés au joueur (donc au champion) qui les a réalisés.

Plutôt que deviner à la main quelle catégorie s'applique à quel champ, on le **détecte depuis nos propres données** : si les 5 valeurs d'un champ sont toujours identiques au sein d'une équipe, sur tout le dataset, c'est un champ équipe ; sinon, un champ joueur.

**Avertissement** : sur seulement 20 matchs, cette classification est bruitée (un champ réellement "par joueur" peut sembler constant par coïncidence sur un petit échantillon). Elle se refera automatiquement, et deviendra fiable, une fois la vraie collecte lancée.

In [6]:
EXCLUDED_CHALLENGE_PREFIXES = ("SWARM_",)  # Arena
EXCLUDED_CHALLENGE_FIELDS = {"poroExplosions", "fistBumpParticipation", "snowballsHit"}  # ARAM


def is_relevant_challenge_field(field):
    if any(field.startswith(prefix) for prefix in EXCLUDED_CHALLENGE_PREFIXES):
        return False
    return field not in EXCLUDED_CHALLENGE_FIELDS


def to_hashable(value):
    return tuple(value) if isinstance(value, list) else value


team_constant_fields = set()
player_level_fields = set()
list_valued_fields = set()

for raw in raw_df["raw_json"]:
    match_data = json.loads(raw)
    participants = match_data["info"]["participants"]

    for team_id in (100, 200):
        team_players = [p for p in participants if p["teamId"] == team_id]
        challenge_keys = set()
        for p in team_players:
            challenge_keys |= set(p.get("challenges", {}).keys())

        for field in challenge_keys:
            if not is_relevant_challenge_field(field):
                continue

            raw_values = [p.get("challenges", {}).get(field) for p in team_players]
            if any(isinstance(v, list) for v in raw_values):
                list_valued_fields.add(field)
                continue

            values = {to_hashable(v) for v in raw_values}
            if len(values) == 1:
                team_constant_fields.add(field)
            else:
                player_level_fields.add(field)

# un champ n'est "équipe" que s'il n'a JAMAIS varié sur tout le dataset
team_constant_fields -= player_level_fields
team_constant_fields.discard("gameLength")  # doublon avec game_duration, déjà extrait

print("Champs équipe:", len(team_constant_fields))
print("Champs joueur (à sommer):", len(player_level_fields))
print("Champs à valeur liste:", sorted(list_valued_fields))

Champs équipe: 19
Champs joueur (à sommer): 108
Champs à valeur liste: ['legendaryItemUsed']


## Concept 5 — Une ligne par (match, joueur), pas par match

Contrairement à une première version où on agrégeait tout au niveau équipe (erreur corrigée ici), le dataset final doit avoir **une ligne par joueur dans un match** — donc ~10 lignes par match, ~200 pour nos 20 matchs actuels. Chaque ligne garde :
- `champion` : le champion joué sur cette ligne — devient une vraie feature du modèle, plus une simple clé d'agrégation.
- Les **champs joueur**, tels quels, non sommés — attachés à ce champion précis.
- Les **champs équipe**, comme contexte partagé (répétés pour les 5 lignes de la même équipe).
- `team_win` : est-ce que l'équipe de ce joueur a gagné.
- `game_duration` : repris de `matches_df` (déjà filtré des remakes), comme contexte supplémentaire.

Le champ à valeur liste (`legendaryItemUsed`) redevient simple : plus besoin de sommer sur l'équipe, juste le nombre d'objets légendaires utilisés par **ce** joueur (`len(liste)`).

In [7]:
def extract_player_row(participant, match_id, game_duration):
    challenges = participant.get("challenges", {})

    row = {
        "match_id": match_id,
        "team": "blue" if participant["teamId"] == 100 else "red",
        "champion": participant["championName"],
        "team_win": participant["win"],
        "game_duration": game_duration,
    }

    for field in team_constant_fields:
        row[field] = challenges.get(field)

    for field in player_level_fields:
        row[field] = challenges.get(field)

    for field in list_valued_fields:
        row[field] = len(challenges.get(field) or [])

    return row


# game_duration déjà filtré des remakes dans matches_df (Concept 3)
game_duration_by_match = dict(zip(matches_df["match_id"], matches_df["game_duration"]))
raw_json_by_match = dict(zip(raw_df["match_id"], raw_df["raw_json"]))

player_rows = []
for match_id, game_duration in game_duration_by_match.items():
    match_data = json.loads(raw_json_by_match[match_id])
    for participant in match_data["info"]["participants"]:
        player_rows.append(extract_player_row(participant, match_id, game_duration))

dataset_df = pd.DataFrame(player_rows)
print("Dataset final:", dataset_df.shape)
dataset_df.head()

Dataset final: (123460, 133)


,match_id,team,champion,team_win,game_duration,baronBuffGoldAdvantageOverThreshold,hadOpenNexus,firstTurretKilled,multiTurretRiftHeraldCount,teamBaronKills,...,killParticipation,earliestDragonTakedown,visionScoreAdvantageLaneOpponent,mejaisFullStackInTime,wardsGuarded,deathsByEnemyChamps,goldPerMinute,baronTakedowns,bountyGold,legendaryItemUsed
0,EUW1_7981560267,blue,Jax,True,1141,NaN,0,0,0,0,...,0.153846,NaN,-0.218590,0,0,6,322.089398,0,0.000000,1
1,EUW1_7981560267,blue,Shyvana,True,1141,NaN,0,0,0,0,...,0.307692,392.037279,-0.005620,0,0,1,446.268819,0,0.000000,2
2,EUW1_7981560267,blue,Leblanc,True,1141,NaN,0,0,0,0,...,0.730769,NaN,0.659407,0,0,4,594.502902,0,533.088348,2
3,EUW1_7981560267,blue,Yunara,True,1141,NaN,0,0,0,0,...,0.615385,NaN,-0.324090,0,0,3,559.798420,0,0.000000,2
4,EUW1_7981560267,blue,Leona,True,1141,NaN,0,0,0,0,...,0.692308,NaN,0.439405,0,0,1,329.371611,0,0.000000,2


---

**Checkpoint** : `dataset_df` a maintenant **123 460 lignes** (12 346 matchs `CLASSIC` valides × 10 joueurs) et **133 colonnes**, sur la collecte à grande échelle du 2026-09-15/16. 173 champions distincts représentés, le moins fréquent avec 93 occurrences — largement suffisant pour la suite.

Avant l'entraînement du modèle : une étape d'exploration visuelle dans Power BI (voir section suivante), puis entraînement GBT + SHAP dans une prochaine session.

## Concept 6 — Export pour exploration visuelle dans Power BI

Avant l'entraînement du modèle, une étape d'exploration visuelle interactive dans **Power BI** (Desktop, sur une autre machine Windows) : regarder les taux de victoire par champion, les corrélations entre faits de jeu et victoire, la distribution des features, etc. — avant de se lancer dans le ML.

Power BI ne tourne pas sur cette machine (Linux) — on exporte donc `dataset_df` en **CSV**, à importer côté Windows. On garde le format plat (une ligne par match/joueur, comme construit) plutôt que de pré-agréger en Python : Power BI est justement fait pour construire des pivots/agrégations interactifs (taux de victoire par champion, par tranche de `killParticipation`...) sans réécrire de code à chaque nouvelle question — pré-agréger maintenant limiterait ce qu'on peut explorer ensuite.

In [8]:
import os

export_path = "../data/dataset_for_powerbi.csv"
dataset_df.to_csv(export_path, index=False)

size_mb = os.path.getsize(export_path) / (1024 * 1024)
print(f"Exporté : {export_path} ({size_mb:.1f} Mo, {dataset_df.shape[0]} lignes, {dataset_df.shape[1]} colonnes)")

Exporté : ../data/dataset_for_powerbi.csv (57.7 Mo, 123460 lignes, 133 colonnes)


---

**Checkpoint** : confirme la taille du fichier exporté, puis transfère `data/dataset_for_powerbi.csv` vers ta machine Windows (le fichier n'est pas versionné dans Git, comme `matches.db`).

Quelques pistes pour démarrer l'exploration dans Power BI, une fois importé :
- **Taux de victoire par champion** (`champion` en axe, `team_win` moyenné) — un premier aperçu direct, à croiser ensuite avec les faits de jeu.
- **Corrélation faits de jeu ↔ victoire** : pour quelques champs clés (`killParticipation`, `teamBaronKills`, `laningPhaseGoldExpAdvantage`...), comparer leur distribution entre lignes `team_win = True` et `team_win = False`.
- **Qualité des données** : vérifier s'il reste des valeurs manquantes (`NaN`, visibles dans l'aperçu `dataset_df.head()` plus haut, ex: `killsOnOtherLanesEarlyJungleAsLaner`) — certains champs `challenges` ne sont pas calculés dans toutes les situations.

Une fois cette exploration faite, prochaine étape technique : entraîner le modèle Gradient Boosted Trees et l'expliquer avec SHAP.